# Active-Learning Capture Selection (PANAMA-style)

This notebook mounts Google Drive, copies one folder from `MyDrive` locally, rewrites the audio references in your data config so they resolve against the copied Colab files, installs the training repo from either GitHub or a zip in Drive (filtering macOS archive metadata), starts TensorBoard, and runs **one active-learning round** via `scripts/active_learn.py` with live terminal-style output.

Each round trains a disposable 4-member **ConcatLSTM** ensemble, searches the control space for the settings where the members **disagree most** (query by committee), and writes the next set of recommended captures. TensorBoard shows the training of every ensemble member.

See [`docs/active_learning_usage.md`](https://github.com/phillipmself/neural-amp-modeler-parametric/blob/feature/active-learning/docs/active_learning_usage.md) for the full workflow.

## What to put in Drive

Put one folder in Google Drive that contains:

- `input.wav` — the fixed reamp input clip
- your reamped output wav files (for round 0, the `starter_*.wav` captures referenced by `data.json`)
- `data.json` — the parametric data config (round-0 seed)
- `model.json` — the ConcatLSTM ensemble-member config (`net.name = "ConcatLSTM"`)
- `learning.json` — the ensemble-member learning config
- optionally, a repo zip such as `nam_parametric_repo.zip` (a Finder-created macOS zip is fine)

In the setup cell, enter the folder path relative to `MyDrive/`, e.g. `parametric-training/amp_captures`.

### Running successive rounds

This notebook runs **one round per execution**, controlled by `ROUND_IDX`:

1. **Round 0** — leave `ROUND_IDX = 0` and `DATA_CONFIG_NAME = "data.json"`. Run the cells. The round trains the ensemble and writes recommendations to `active_learning_output/` (synced back to your Drive folder).
2. **Reamp** — `proposed_captures_round_0.json` lists the recommended settings with suggested wav filenames (e.g. `round_0_00.wav`). Reamp `input.wav` at each setting, save the wavs, and add them to your Drive folder.
3. **Fill in** — open `active_learning_output/aggregated_data_config_0.json` (synced to Drive); its appended train entries have placeholder `y_path`s pointing at those new wavs. Put the wavs in place so the names match.
4. **Round 1** — copy `aggregated_data_config_0.json` into your Drive folder, set `DATA_CONFIG_NAME` to its filename and `ROUND_IDX = 1`, and rerun. Repeat to grow the capture set.

For `ROUND_IDX > 0` the driver prints a warning that an explicit `--data-config` bypasses the auto-discovered previous aggregated config — that is expected here, because the filled aggregated config you point at already contains all accumulated captures.

In [ ]:
from pathlib import Path

DRIVE_FOLDER = "<folder path, do not include MyDrive>".strip("/")
if not DRIVE_FOLDER:
    raise ValueError("Set DRIVE_FOLDER to a folder under MyDrive, without a leading 'MyDrive/'.")

# Which round this is. 0 seeds from your starter data.json; for round N > 0 point
# DATA_CONFIG_NAME at the filled aggregated_data_config_{N-1}.json from the previous round.
ROUND_IDX = 0

DATA_CONFIG_NAME = "data.json"
MODEL_CONFIG_NAME = "model.json"
LEARNING_CONFIG_NAME = "learning.json"

# Active-learning round knobs (see scripts/active_learn.py --help).
ENSEMBLE_SIZE = 4        # disposable ConcatLSTM members (see MAX_WORKERS for parallelism)
MAX_WORKERS = None       # members to train concurrently. None = serial (one member at a
                         # time). The ConcatLSTM is tiny and underutilizes a big GPU, so on
                         # a single large-VRAM card (e.g. RTX 6000) with a small batch size
                         # you can set this to ENSEMBLE_SIZE to train them all at once for a
                         # ~2-3x speedup. Peak memory scales with the worker count x
                         # (batch_size x ny) per member -- lower it if you OOM.
NUM_RESTARTS = 14        # random latent inits per switch combination during g-opt.
                         # This is what GENERATES candidates: the pool is
                         # NUM_RESTARTS x (num switch combos), then clustering/dedup only
                         # shrinks it. To reliably fill MAX_PER_ROUND, set this comfortably
                         # above MAX_PER_ROUND (e.g. 16-20) so duplicates can be merged away.
NUM_STEPS = 200          # Adam ascent steps per restart during g-opt (PANAMA uses 300)
MAX_PER_ROUND = 10        # CAP on recommended captures emitted this round (not a target)
G_OPT_NY = 4096         # g-opt window length (samples per chunk). Cost is ~linear in this;
                         # smaller windows = faster search. PANAMA uses 8192.
G_OPT_BATCH_SIZE = 64    # g-opt batch size (chunks per step)
G_OPT_LR = 0.02          # Adam learning rate for the g-opt latent ascent (PANAMA uses 0.02)
USE_MEL = False          # add PANAMA's multi-resolution mel-variance term
SEED = 0                 # seeds ensemble member inits and g-opt latent inits

# NOTE: on a CUDA GPU the g-opt search auto-selects a fast cuDNN path (members run in
# train() mode with cuDNN enabled) whenever it is provably equivalent to eval() -- i.e.
# the model has train_truncate=None and no dropout/batchnorm. The round log prints which
# path was taken. If you see the slow "eval() + cuDNN-disabled" message, set
# train_truncate=None in model.json (and drop any dropout/batchnorm) for the ~10x speedup.

REPO_SOURCE = "github"  # "github" or "zip"
REPO_ZIP_NAME = "nam_parametric_repo.zip"  # used only when REPO_SOURCE == "zip"

REPO_URL = "https://github.com/phillipmself/neural-amp-modeler-parametric.git"
REPO_BRANCH = "feature/active-learning"

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_DIR = DRIVE_ROOT / DRIVE_FOLDER
LOCAL_ROOT = Path("/content/nam_parametric")
LOCAL_DATA_ROOT = LOCAL_ROOT / "data"
REPO_DIR = LOCAL_ROOT / "repo"
# Ensemble checkpoints, member TensorBoard logs, proposals, and aggregated configs
# all land here (one subtree per round). This is also the TensorBoard log dir.
OUTPUT_DIR = LOCAL_ROOT / "active_learning_output"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FOLDER_NAME = Path(DRIVE_FOLDER).name
LOCAL_DATA_DIR = LOCAL_DATA_ROOT / LOCAL_FOLDER_NAME
DRIVE_OUTPUT_ROOT = DRIVE_DIR / "active_learning_output"
SYNC_INTERVAL_SECONDS = 60

print(f"Drive folder: MyDrive/{DRIVE_FOLDER}")
print(f"Round index: {ROUND_IDX}")
print(f"Repo source: {REPO_SOURCE}")
if REPO_SOURCE == "zip":
    print(f"Repo zip: {REPO_ZIP_NAME}")
print(f"Local data dir: {LOCAL_DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Drive output dir: {DRIVE_OUTPUT_ROOT}")

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import PurePosixPath

def _is_archive_metadata_name(name: str) -> bool:
    return name in {"__MACOSX", ".DS_Store"} or name.startswith(".") or name.startswith("._")

def _is_archive_metadata_path(path: Path | PurePosixPath) -> bool:
    return any(_is_archive_metadata_name(part) for part in path.parts if part not in {"", "."})

def _looks_like_repo_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").exists()
        and (path / "scripts" / "active_learn.py").exists()
        and (path / "nam" / "train" / "active_learning.py").exists()
    )

def _resolve_repo_source_dir(extract_root: Path) -> Path:
    if _looks_like_repo_root(extract_root):
        return extract_root
    candidates = sorted(
        {
            path.parent
            for path in extract_root.rglob("pyproject.toml")
            if not _is_archive_metadata_path(path.relative_to(extract_root))
        },
        key=lambda path: (len(path.relative_to(extract_root).parts), str(path)),
    )
    for candidate in candidates:
        if _looks_like_repo_root(candidate):
            return candidate
    raise FileNotFoundError(
        "Couldn't find the repo root in the extracted archive. Expected pyproject.toml, "
        f"scripts/active_learn.py and nam/train/active_learning.py under {extract_root}"
    )

def _resolve_repo_zip_path() -> Path:
    if not REPO_ZIP_NAME:
        raise ValueError("Set REPO_ZIP_NAME when REPO_SOURCE is 'zip'")
    direct_path = DRIVE_DIR / REPO_ZIP_NAME
    if direct_path.exists():
        return direct_path
    matches = sorted(
        path
        for path in DRIVE_DIR.rglob(REPO_ZIP_NAME)
        if not _is_archive_metadata_path(path.relative_to(DRIVE_DIR))
    )
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f"Repo zip not found under: {DRIVE_DIR}")
    raise FileExistsError(f"Found multiple repo zips named {REPO_ZIP_NAME!r} under {DRIVE_DIR}")

def _extract_repo_zip(repo_zip_path: Path, extract_root: Path) -> None:
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    kept_members = []
    with zipfile.ZipFile(repo_zip_path) as archive:
        for info in archive.infolist():
            relative_path = PurePosixPath(info.filename)
            if _is_archive_metadata_path(relative_path):
                continue
            if any(part == ".." for part in relative_path.parts):
                raise ValueError(f"Unsafe path inside repo zip: {info.filename}")
            archive.extract(info, extract_root)
            kept_members.append(info.filename)
    if not kept_members:
        raise FileNotFoundError(f"No usable files found in repo zip: {repo_zip_path}")

extract_root = LOCAL_ROOT / "repo_extract"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if extract_root.exists():
    shutil.rmtree(extract_root)

if REPO_SOURCE == "github":
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif REPO_SOURCE == "zip":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    repo_zip_path = _resolve_repo_zip_path()
    if not zipfile.is_zipfile(repo_zip_path):
        raise ValueError(f"Repo archive must be a zip file: {repo_zip_path}")
    _extract_repo_zip(repo_zip_path, extract_root)
    repo_source_dir = _resolve_repo_source_dir(extract_root)
    shutil.move(str(repo_source_dir), str(REPO_DIR))
else:
    raise ValueError("REPO_SOURCE must be 'github' or 'zip'")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

import torch
import pytorch_lightning as pl
import tensorboard

print(f"Resolved repo directory: {REPO_DIR}")
print(f"Torch: {torch.__version__}")
print(f"PyTorch Lightning: {pl.__version__}")
print(f"TensorBoard: {tensorboard.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Installed repo from: {REPO_SOURCE}")

In [ ]:
import json
from pathlib import PurePosixPath, PureWindowsPath
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

if not DRIVE_DIR.exists():
    raise FileNotFoundError(f"Drive folder not found: {DRIVE_DIR}")

def _ignore_drive_copy(dirpath: str, names: list[str]) -> set[str]:
    current_dir = Path(dirpath)
    return {
        name
        for name in names
        if _is_archive_metadata_name(name)
        or (current_dir == DRIVE_DIR and name == "active_learning_output")
        or (REPO_SOURCE == "zip" and name == REPO_ZIP_NAME)
    }

def _sync_tree(source_root: Path, dest_root: Path) -> None:
    dest_root.mkdir(parents=True, exist_ok=True)
    for path in source_root.rglob("*"):
        relative = path.relative_to(source_root)
        target = dest_root / relative
        if path.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            src_stat = path.stat()
            dst_stat = target.stat()
            if (
                src_stat.st_size == dst_stat.st_size
                and src_stat.st_mtime_ns <= dst_stat.st_mtime_ns
            ):
                continue
        shutil.copy2(path, target)

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
shutil.copytree(DRIVE_DIR, LOCAL_DATA_DIR, ignore=_ignore_drive_copy)

# Restore prior rounds' output (ensemble logs, aggregated configs) so TensorBoard shows
# every member trained so far and re-runs don't lose earlier rounds.
if DRIVE_OUTPUT_ROOT.exists():
    _sync_tree(DRIVE_OUTPUT_ROOT, OUTPUT_DIR)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_config_path = LOCAL_DATA_DIR / DATA_CONFIG_NAME
model_config_path = LOCAL_DATA_DIR / MODEL_CONFIG_NAME
learning_config_path = LOCAL_DATA_DIR / LEARNING_CONFIG_NAME

for path in (data_config_path, model_config_path, learning_config_path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def _load_json_config(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path.name} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc

data_config = _load_json_config(data_config_path)
model_config = _load_json_config(model_config_path)
learning_config = _load_json_config(learning_config_path)

net_name = model_config.get("net", {}).get("name")
if net_name != "ConcatLSTM":
    raise ValueError(
        f"{MODEL_CONFIG_NAME} must have net.name == 'ConcatLSTM' for the active-learning "
        f"ensemble; got {net_name!r}"
    )

def _candidate_local_audio_paths(raw_path: str) -> list[Path]:
    candidates = []
    posix_path = PurePosixPath(raw_path)
    windows_path = PureWindowsPath(raw_path)

    def _append(candidate: Path) -> None:
        if candidate not in candidates:
            candidates.append(candidate)

    if not windows_path.is_absolute() and not posix_path.is_absolute():
        relative_parts = [part for part in posix_path.parts if part not in {"", "."}]
        if relative_parts:
            _append(LOCAL_DATA_DIR / Path(*relative_parts))

    basename = windows_path.name or posix_path.name
    if basename:
        _append(LOCAL_DATA_DIR / basename)

    for pure_path in (windows_path, posix_path):
        if pure_path.is_absolute():
            anchor = pure_path.anchor
            relative_parts = [part for part in pure_path.parts if part not in {"", ".", anchor}]
            if relative_parts:
                _append(LOCAL_DATA_DIR / Path(*relative_parts))

    return candidates

def _rewrite_audio_path(raw_path: str) -> str:
    candidates = _candidate_local_audio_paths(raw_path)
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    attempted = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        f"Couldn't map audio path {raw_path!r} into the copied Drive folder {LOCAL_DATA_DIR}.\n"
        "For round N > 0 this usually means a recommended capture has not been reamped yet: "
        "record the proposed wavs and add them to your Drive folder.\n"
        f"Tried:\n{attempted}"
    )

resolved_audio_paths: set[str] = set()

def _rewrite_path_fields(mapping: dict) -> None:
    for key in ("x_path", "y_path"):
        raw_path = mapping.get(key)
        if raw_path is None:
            continue
        resolved_path = _rewrite_audio_path(raw_path)
        mapping[key] = resolved_path
        resolved_audio_paths.add(resolved_path)

common = data_config.get("common", {})
if isinstance(common, dict):
    _rewrite_path_fields(common)

def _rewrite_section(section_name: str) -> None:
    section = data_config.get(section_name)
    if isinstance(section, dict):
        _rewrite_path_fields(section)
        return
    if isinstance(section, list):
        for item in section:
            if isinstance(item, dict):
                _rewrite_path_fields(item)

for section_name in ("train", "validation"):
    _rewrite_section(section_name)

with open(data_config_path, "w") as fp:
    json.dump(data_config, fp, indent=4)

# g-opt reamps one fixed input clip; default it to the data config's own input.
G_OPT_INPUT_WAV = common.get("x_path") if isinstance(common, dict) else None

print("Copied folder locally and rewrote config audio paths.")
print(f"Resolved {len(resolved_audio_paths)} audio file paths.")
print(f"Data config: {data_config_path}")
print(f"Model config: {model_config_path}")
print(f"Learning config: {learning_config_path}")
print(f"g-opt input wav: {G_OPT_INPUT_WAV}")
print(f"Round artifacts will sync to: {DRIVE_OUTPUT_ROOT}")

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR

In [ ]:
import errno
import os
import pty
import select
import time

def _sync_output_to_drive() -> None:
    _sync_tree(OUTPUT_DIR, DRIVE_OUTPUT_ROOT)

command = [
    sys.executable,
    "scripts/active_learn.py",
    "--round-idx", str(ROUND_IDX),
    "--output-dir", str(OUTPUT_DIR),
    "--data-config", str(data_config_path),
    "--model-config", str(model_config_path),
    "--learning-config", str(learning_config_path),
    "--ensemble-size", str(ENSEMBLE_SIZE),
    "--num-restarts", str(NUM_RESTARTS),
    "--num-steps", str(NUM_STEPS),
    "--max-per-round", str(MAX_PER_ROUND),
    "--g-opt-ny", str(G_OPT_NY),
    "--g-opt-batch-size", str(G_OPT_BATCH_SIZE),
    "--g-opt-lr", str(G_OPT_LR),
    "--seed", str(SEED),
]
if MAX_WORKERS:
    command += ["--max-workers", str(MAX_WORKERS)]
if G_OPT_INPUT_WAV:
    command += ["--g-opt-input-wav", str(G_OPT_INPUT_WAV)]
if USE_MEL:
    command += ["--use-mel"]

print("Running: " + " ".join(command))
print()

master_fd, slave_fd = pty.openpty()
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    stdin=subprocess.DEVNULL,
    stdout=slave_fd,
    stderr=slave_fd,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
    text=False,
)
os.close(slave_fd)

last_sync = time.monotonic()

try:
    while True:
        ready, _, _ = select.select([master_fd], [], [], 0.1)
        if master_fd in ready:
            try:
                chunk = os.read(master_fd, 4096)
            except OSError as exc:
                if exc.errno == errno.EIO:
                    break
                raise
            if chunk:
                print(chunk.decode("utf-8", errors="replace"), end="")
            else:
                break
        now = time.monotonic()
        if now - last_sync >= SYNC_INTERVAL_SECONDS:
            _sync_output_to_drive()
            last_sync = now
        if process.poll() is not None and master_fd not in ready:
            break
finally:
    os.close(master_fd)
    _sync_output_to_drive()

return_code = process.wait()
_sync_output_to_drive()
if return_code != 0:
    raise RuntimeError(f"Active-learning round failed with exit code {return_code}")

print()
print(f"Round {ROUND_IDX} complete. Output synced to: {DRIVE_OUTPUT_ROOT}")

In [ ]:
import json
from IPython.display import Image, display

proposals_path = OUTPUT_DIR / f"proposed_captures_round_{ROUND_IDX}.json"
aggregated_path = OUTPUT_DIR / f"aggregated_data_config_{ROUND_IDX}.json"
plot_path = OUTPUT_DIR / f"accepted_capture_distributions_round_{ROUND_IDX}.png"

if not proposals_path.exists():
    raise RuntimeError(f"No proposals found at {proposals_path}; run the round cell first.")

proposals = json.loads(proposals_path.read_text())

print(f"=== Recommended captures for round {ROUND_IDX} ({len(proposals)}) ===\n")
if not proposals:
    print("(none — the disagreement search returned no new settings this round)")
for index, record in enumerate(proposals, start=1):
    settings = ", ".join(f"{name}={value}" for name, value in record["params"].items())
    score = record.get("score")
    score_str = f"  [disagreement {score:.4g}]" if isinstance(score, (int, float)) else ""
    print(f"{index}. {record['y_path']} -> {settings}{score_str}")

print(f"\nProposals file: {proposals_path}")
print(f"Aggregated config for next round: {aggregated_path}")
print("\nNext: reamp input.wav at each setting above, save the wavs under the listed names")
print("in your Drive folder, then run the next round with ROUND_IDX + 1 pointing")
print(f"DATA_CONFIG_NAME at aggregated_data_config_{ROUND_IDX}.json.")

print(f"\n=== Round {ROUND_IDX} output files (also synced to Drive) ===")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))

if plot_path.exists():
    print("\nAccepted-capture distribution (train set after this round):")
    display(Image(filename=str(plot_path)))

In [ ]:
# === Re-emit MORE captures from the already-trained ensemble (no retraining) ===
# Reuses the checkpoints from the round you just ran and only re-does the g-opt
# search + selection. Keep SEED the same as the original run so the result is a
# clean superset of the proposals you already got.

NEW_MAX_PER_ROUND = 10          # how many captures to emit now (was MAX_PER_ROUND)
NEW_NUM_RESTARTS = NUM_RESTARTS  # raise this (e.g. 16-20) if you don't get enough distinct
                                 # settings -- restarts generate candidates, MAX_PER_ROUND
                                 # is only the cap. The round warns when it falls short.

import errno, os, pty, select, sys, subprocess, time

# Sanity: earlier cells must have run in this kernel.
for _name in ("REPO_DIR", "OUTPUT_DIR", "ROUND_IDX", "data_config_path",
              "model_config_path", "learning_config_path", "_sync_tree", "DRIVE_OUTPUT_ROOT"):
    if _name not in globals():
        raise RuntimeError(f"{_name} is not defined — run the setup/round cells above first.")

ensemble_dir = OUTPUT_DIR / f"ensemble_round_{ROUND_IDX}"
ckpts = sorted(ensemble_dir.glob("member_*/best.ckpt"))
if not ckpts:
    raise FileNotFoundError(
        f"No member checkpoints found under {ensemble_dir}. The training phase must have "
        "completed (each member writes best.ckpt only after it finishes). If you trained in a "
        "previous session, make sure DRIVE_OUTPUT_ROOT synced back into OUTPUT_DIR."
    )
print(f"Reusing {len(ckpts)} checkpoints:")
for c in ckpts:
    print(f"  {c}")

def _sync_output_to_drive():
    _sync_tree(OUTPUT_DIR, DRIVE_OUTPUT_ROOT)

command = [
    sys.executable, "scripts/active_learn.py",
    "--round-idx", str(ROUND_IDX),
    "--output-dir", str(OUTPUT_DIR),
    "--data-config", str(data_config_path),
    "--model-config", str(model_config_path),
    "--learning-config", str(learning_config_path),
    "--ckpts", *[str(c) for c in ckpts],
    "--max-per-round", str(NEW_MAX_PER_ROUND),
    "--num-restarts", str(NEW_NUM_RESTARTS),
    "--num-steps", str(NUM_STEPS),
    "--g-opt-ny", str(G_OPT_NY),
    "--g-opt-batch-size", str(G_OPT_BATCH_SIZE),
    "--g-opt-lr", str(G_OPT_LR),
    "--seed", str(SEED),
]
if G_OPT_INPUT_WAV:
    command += ["--g-opt-input-wav", str(G_OPT_INPUT_WAV)]
if USE_MEL:
    command += ["--use-mel"]

print("\nRunning: " + " ".join(command) + "\n")

master_fd, slave_fd = pty.openpty()
process = subprocess.Popen(
    command, cwd=REPO_DIR, stdin=subprocess.DEVNULL,
    stdout=slave_fd, stderr=slave_fd,
    env={**os.environ, "PYTHONUNBUFFERED": "1"}, text=False,
)
os.close(slave_fd)
last_sync = time.monotonic()
try:
    while True:
        ready, _, _ = select.select([master_fd], [], [], 0.1)
        if master_fd in ready:
            try:
                chunk = os.read(master_fd, 4096)
            except OSError as exc:
                if exc.errno == errno.EIO:
                    break
                raise
            if chunk:
                print(chunk.decode("utf-8", errors="replace"), end="")
            else:
                break
        if time.monotonic() - last_sync >= SYNC_INTERVAL_SECONDS:
            _sync_output_to_drive()
            last_sync = time.monotonic()
        if process.poll() is not None and master_fd not in ready:
            break
finally:
    os.close(master_fd)
    _sync_output_to_drive()

return_code = process.wait()
_sync_output_to_drive()
if return_code != 0:
    raise RuntimeError(f"Re-emit failed with exit code {return_code}")
print(f"\nRe-emitted up to {NEW_MAX_PER_ROUND} captures for round {ROUND_IDX}. "
      f"Synced to {DRIVE_OUTPUT_ROOT}")
